In [1]:
import pandas as pd

# 1. 讀取資料
df_raw = pd.read_csv('YRBS_2007.csv')

# 【修改】加入年齡欄位，這裡假設欄位名稱為 'HowOldAreYou' 
# (如果你的資料集裡叫 'Age'，請自行替換)
cols = ['WhatIsYourSex', 'CurrentAlcoholUse', 'HowOldAreYou']
df_subset = df_raw[cols].copy()

print("--- 原始資料前五筆 ---")
print(df_subset.head())

print("\n--- 清理前缺失值統計 ---")
print(df_subset.isnull().sum())

--- 原始資料前五筆 ---
   WhatIsYourSex  CurrentAlcoholUse  HowOldAreYou
0            2.0                NaN           4.0
1            2.0                NaN           7.0
2            2.0                NaN           NaN
3            1.0                1.0           1.0
4            1.0                NaN           1.0

--- 清理前缺失值統計 ---
WhatIsYourSex          13
CurrentAlcoholUse    1372
HowOldAreYou           61
dtype: int64


In [3]:
# 2. 刪除含有缺失值的資料
df_cleaned = df_subset.dropna().copy()
print(f"\n刪除缺失值後的樣本數：{len(df_cleaned)}")


刪除缺失值後的樣本數：12615


In [5]:
# 3. 變數重編碼 (Recoding)

# 飲酒習慣重編碼：2-7 -> 1 (Yes), 1 -> 0 (No)
def recode_alcohol(code):
    if code == 1:
        return 0
    elif 2 <= code <= 7:
        return 1
    return None

# 性別重編碼：1 -> 1 (Female), 2 -> 0 (Male)
def recode_sex(code):
    if code == 1:
        return 1  # Female
    elif code == 2:
        return 0  # Male
    return None

# 【新增】年齡重編碼：依據 YRBS 官方代碼轉換為實際年齡數字
# 通常代碼為：1 = 12歲以下, 2 = 13歲, 3 = 14歲, 4 = 15歲, 5 = 16歲, 6 = 17歲, 7 = 18歲以上
def recode_age(code):
    mapping = {
        1: 12,  # 或 12 歲以下
        2: 13,
        3: 14,
        4: 15,
        5: 16,
        6: 17,
        7: 18   # 或 18 歲以上
    }
    return mapping.get(code, None)

# 執行重編碼
df_cleaned['Alcohol_Binary'] = df_cleaned['CurrentAlcoholUse'].apply(recode_alcohol)
df_cleaned['Sex_Binary'] = df_cleaned['WhatIsYourSex'].apply(recode_sex)
df_cleaned['Age_Numeric'] = df_cleaned['HowOldAreYou'].apply(recode_age) # 【新增】

In [7]:
# 4. 篩選最終欄位並確保沒有因重編碼產生的缺失值
df_final = df_cleaned[['Sex_Binary', 'Alcohol_Binary', 'Age_Numeric']].dropna()

In [9]:
# 5. 統計與儲存
print("\n--- 重編碼後的結果統計 (按性別與年齡看飲酒比例) ---")
# 這裡多加入了 Age_Numeric 讓分組更詳細
print(df_final.groupby(['Sex_Binary', 'Age_Numeric'])['Alcohol_Binary'].value_counts(normalize=True))

# 儲存到 processed 資料夾
import os
os.makedirs('data/processed', exist_ok=True) # 確保資料夾存在
df_final.to_csv('data/processed/yrbs_cleaned.csv', index=False)
print("\n已成功儲存至 data/processed/yrbs_cleaned.csv")


--- 重編碼後的結果統計 (按性別與年齡看飲酒比例) ---
Sex_Binary  Age_Numeric  Alcohol_Binary
0           12           1                 0.833333
                         0                 0.166667
            13           0                 0.666667
                         1                 0.333333
            14           0                 0.727731
                         1                 0.272269
            15           0                 0.621986
                         1                 0.378014
            16           0                 0.536862
                         1                 0.463138
            17           1                 0.527181
                         0                 0.472819
            18           1                 0.555340
                         0                 0.444660
1           12           1                 0.833333
                         0                 0.166667
            13           1                 0.666667
                         0                 